In [ ]:
import os
import sys
import json
import random
import shutil
import platform
import subprocess
import numpy as np
import torch

print("Libraries imported successfully.")

In [ ]:
SEED = 42
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print(f"Random seed set to {SEED}.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
gpu_name = None
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Model: {gpu_name}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
DIRS = {
    "data_dir": os.path.join(BASE_DIR, "data"),
    "features_dir": os.path.join(BASE_DIR, "features"),
    "checkpoints_dir": os.path.join(BASE_DIR, "checkpoints"),
    "results_dir": os.path.join(BASE_DIR, "results"),
    "audio_wav_dir": os.path.join(BASE_DIR, "data", "audio_wav"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
    print(f"Verified/Created directory: {d}")

In [ ]:
import hashlib, datetime

LEGACY_BACKUP = os.path.join(BASE_DIR, "legacy_backup_eGeMAPS_DenseNet")
_old_cfg_path = os.path.join(BASE_DIR, "config.json")

def _sha1(path):
    h = hashlib.sha1()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def _legacy_pipeline_present():
    if not os.path.exists(_old_cfg_path):
        return False
    try:
        old = json.load(open(_old_cfg_path))
    except Exception:
        return True                                  
    return old.get("audio_feature_set") == "eGeMAPSv02" or old.get("video_backbone") == "densenet121"

def backup_legacy_results():
    if os.path.exists(LEGACY_BACKUP):
        print(f"Legacy backup already exists -> left untouched: {LEGACY_BACKUP}")
        return
    if not _legacy_pipeline_present():
        print("No legacy (eGeMAPS / DenseNet) pipeline found in this working directory -> no backup needed.")
        return
    tmp = LEGACY_BACKUP + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp)
    items = [f for f in sorted(os.listdir(BASE_DIR)) if os.path.isfile(os.path.join(BASE_DIR, f)) and f.endswith((".json", ".py"))]
    trees = [d for d in ("results", "executed_notebooks", "logs") if os.path.isdir(os.path.join(BASE_DIR, d))]
    manifest = {}
    for f in items:
        shutil.copy2(os.path.join(BASE_DIR, f), os.path.join(tmp, f))
    for d in trees:
        shutil.copytree(os.path.join(BASE_DIR, d), os.path.join(tmp, d))
    for root, _, files in os.walk(tmp):
        for f in files:
            dst = os.path.join(root, f); rel = os.path.relpath(dst, tmp); src = os.path.join(BASE_DIR, rel)
            if _sha1(src) != _sha1(dst):
                shutil.rmtree(tmp, ignore_errors=True)
                raise RuntimeError(f"Backup verification failed for {rel}; nothing was overwritten. Fix the cause (disk space?) and re-run Phase 1.")
            manifest[rel] = {"sha1": _sha1(dst), "bytes": os.path.getsize(dst)}
    with open(os.path.join(tmp, "MANIFEST.json"), "w") as fh:
        json.dump({"created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"), "reason": "pipeline migrated from eGeMAPS audio + DenseNet-121 video to WavLM + face features",
                   "note": "copied byte-for-byte; checkpoints/ and features/ are intentionally not copied (see Phase 1 comments)", "n_files": len(manifest), "files": manifest}, fh, indent=2)
    os.replace(tmp, LEGACY_BACKUP)
    print(f"Legacy backup created and SHA-1 verified: {len(manifest)} files in {LEGACY_BACKUP}")

backup_legacy_results()

In [ ]:
tool_paths = {t: shutil.which(t) for t in ("ffmpeg", "ffprobe")}
missing_tools = [t for t, p in tool_paths.items() if p is None]
if missing_tools:
    raise RuntimeError(
        f"Required tools not found on PATH: {missing_tools}. "
        "Install ffmpeg (e.g. `apt-get install -y ffmpeg`) before running Phases 4-5."
    )
ffmpeg_version = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
print(f"Found: {tool_paths}")
print(ffmpeg_version)

In [ ]:
config = {
    "seed": SEED,
    "base_dir": BASE_DIR,
    "dataset": "MELD",
    "num_classes": 7,
    "emotions": ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"],
    "modalities": ["text", "audio", "video"],

    "text_model": "roberta-base",
    "text_feature_dim": 768,
    "audio_feature_set": "wavlm-base-plus:uniform-mean-of-transformer-layers-1..12",   
    "audio_feature_level": "utterance-mean-over-time",
    "audio_model": "microsoft/wavlm-base-plus",
    "audio_feature_dim": 768,         
    "audio_sample_rate": 16000,
    "audio_min_seconds": 0.1,        
    "video_backbone": "mtcnn+hsemotion",  
    "video_hsemotion_model": "enet_b0_8_best_vgaf",
    "video_feature_dim": 1280,        
    "video_num_frames": 8,            
    "video_decode_fps": 4,            
    "video_frame_size": 224,          
    "video_detect_size": [640, 360],  
    "video_min_face_prob": 0.90,      
    "video_min_face_px": 40,          
    "video_max_missing_fraction": 0.30,   

    "mc_dropout_passes": 10,

    "expected_counts": {
        "train": {"utterances": 9989, "dialogues": 1038},
        "dev":   {"utterances": 1109, "dialogues": 114},
        "test":  {"utterances": 2610, "dialogues": 280},
    },

    "media_search_roots": [p for p in os.environ.get("MELD_MEDIA_ROOTS", "/kaggle/input").split(",") if p.strip()],   
    "media_dirs": None,               
    "max_missing_fraction": 0.005,    

    "paths": {k: v for k, v in DIRS.items()},
}

config_path = os.path.join(BASE_DIR, "config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

env_info = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "device": str(device),
    "gpu": gpu_name,
    "ffmpeg": ffmpeg_version,
}
with open(os.path.join(DIRS["results_dir"], "environment.json"), "w") as f:
    json.dump(env_info, f, indent=4)

print(f"Global configuration saved to {config_path}.")

In [ ]:
with open(config_path) as f:
    cfg = json.load(f)
for key in ("audio_feature_dim", "video_feature_dim", "text_feature_dim", "expected_counts", "paths"):
    assert key in cfg, f"Missing config key: {key}"
assert cfg["audio_feature_dim"] == 768 and cfg["video_feature_dim"] == 1280
assert os.path.exists(config_path), "Configuration file missing!"
print("Phase 1 Environment Setup completed successfully.")